In [9]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split

### teach recomendations

combine sibsp and parch into a single feature

a cabin = top floor 
Ticket # = where you boarded from....not sure if this is useful
Embarked = where you boarded from

In [10]:
# Load the dataset
data = pd.read_csv('train.csv')

In [11]:
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [12]:
data.groupby('Survived').count()
#more passangeres dead than alive

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
Survived,,,,,,,,,,,
0,549,549,549,549,424,549,549,549,549,68,549
1,342,342,342,342,290,342,342,342,342,136,340


In [13]:
data.groupby('Pclass').count()
#more passangeres in 3rd class than 1st and 2nd

,PassengerId,Survived,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
Pclass,,,,,,,,,,,
1,216,216,216,216,186,216,216,216,216,176,214
2,184,184,184,184,173,184,184,184,184,16,184
3,491,491,491,491,355,491,491,491,491,12,491


In [14]:
data.groupby('Sex').count()
#more men then women on the ship

,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
Sex,,,,,,,,,,,
female,314,314,314,314,261,314,314,314,314,97,312
male,577,577,577,577,453,577,577,577,577,107,577


After getting a little bit better of an understanding about the titanic dataset, i know now what to look for / might be weird.

For example, there are a lot more men then women on the ship. If the proportion of women saved is higher than the proportion of men, then we knowing something is at play (it's a famous story and all so this is just an example).

Now onto some data analysis

In [15]:
# proportion of survivors
data['Survived'].value_counts(normalize=True)

Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

In [16]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
# Data Preprocessing
# Convert categorical features to numerical
gender_enc = LabelEncoder()
gender_enc.fit(data['Sex'])
data['sex_Bin'] = gender_enc.transform(data['Sex'])


In [18]:
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,sex_Bin
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,1
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,0
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,0
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,1


In [ ]:
# proportion of survivors based on sex
data.groupby('Sex')['Survived'].mean()

# of the survivors, 74% were female.

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

In [ ]:
# proportion of survivors based on Pclass
data.groupby('Pclass')['Survived'].mean()

#1st class passengers had a higher survival rate than 2nd and 3rd class passengers.

Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64

In [49]:
# Dual Income No Children (DINC) = 0
data['DINC'] = 1  

for i in range(len(data)):
    if data.loc[i, 'Parch'] == 0 and data.loc[i, 'SibSp'] == 0:
        data.loc[i, 'DINC'] = 0

In [26]:
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,sex_Bin,DINC
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,1,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,0,0
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,0,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,0,0
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,1,1


In [50]:
data.groupby('DINC')['Survived'].mean()
#I guess if you don't have children, you couldn't use them as a pity factor to get a lifeboat.

DINC
0    0.303538
1    0.505650
Name: Survived, dtype: float64

In [ ]:
# checking cabin information
data['Cabin'].isnull().sum()

#about 687 passengers have no cabin information...not sure if it is worth using

687

In [102]:
data.groupby('Embarked')['Survived'].mean()

Embarked
C    0.553571
Q    0.389610
S    0.336957
Name: Survived, dtype: float64

In [103]:
#if female and a parent = 0
data['FemParent'] = 0  

for i in range(len(data)):
    if data.loc[i, 'Parch'] > 0 and data.loc[i, 'sex_Bin'] == 0:
        data.loc[i, 'FemParent'] = 1

In [ ]:
#if child 
data['child'] = 0  

for i in range(len(data)):
    if data.loc[i, 'Age'] < 10:
        data.loc[i, 'child'] = 1

Overall, the main factors for me ranked are the following:

1. Sex (Almost a huge guaranteor of whether you were going to surive or not)
2. Pclass (if you were 1st class, 60% more likely to survive)
3. DINC has lesser chances of survival...70% of them died vs the 50% that had children

In [124]:
#random forest classifier

from sklearn.ensemble import RandomForestClassifier

In [129]:
# splitting the data into training and testing sets
x = data[['Pclass', 'sex_Bin', 'child', 'FemParent']]
y = data['Survived']


x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.3, random_state=6)

In [ ]:
model = RandomForestClassifier(n_estimators= 100, max_depth=100) # max_dept is the depth of the tree
#changes the number of decisions the tree can make
model.fit(x_train, y_train)

# Evaluate the model
accuracy = model.score(x_test, y_test)
print(f"Accuracy: {accuracy:}")


Accuracy: 0.8470149253731343


I tried DINC and it just made the model worse. It seems to have little importance